In [ ]:
#SPY buy and hold
import pandas as pd
import numpy as np
from empyrical import max_drawdown
import matplotlib.pyplot as plt

def yearly_returns(df):
    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index('Date', inplace=True)
    returns = {}
    for year in range(2019, 2023):
        start_date = f"{year-1}-12-31"
        end_date = f"{year}-12-31"
        start_price = df['Adj Close'].loc[:start_date].iloc[-1]
        end_price = df['Adj Close'].loc[:end_date].iloc[-1]
        yearly_return = (end_price - start_price) / start_price
        returns[year] = yearly_return
    return pd.Series(returns)

data = {
    'spy': pd.read_csv("spy_weekly_return_detailed.csv"),
}

yearly_returns_dict = {key: yearly_returns(df) for key, df in data.items()}
final_df = pd.DataFrame(yearly_returns_dict)
final_df.index.name = 'Date'

cash_annual_bh = []
cash_bh = 100
for year in range(2001, 2024):
    total_returns_bh = cash_bh * (1 + final_df.loc[year])
    cash_bh=total_returns_bh
    cash_annual_bh.append((year, total_returns_bh))

years_w, cash_values_bh = zip(*cash_annual_bh)
data=cash_annual_bh
years = [item[0] for item in data]
values = [item[1]['spy'] for item in data]

df1 = pd.DataFrame({'Year': years, 'Value': values})
initial_value = 100
df1['Annualized_Return'] = (df1['Value'] - df1['Value'].shift(1, fill_value=initial_value)) / df1['Value'].shift(1, fill_value=initial_value)*100


df=pd.read_csv("spy_weekly_return_detailed.csv")
df=df[df["Year"]>=2001]
yearly_volatility = df.groupby('Year')['Return'].std() * np.sqrt(252)

df1 = df1.merge(yearly_volatility.rename('Yearly_Volatility'), on='Year', how='left')

df1["Sharpe_Ratio"]=df1["Annualized_Return"]/df1["Yearly_Volatility"]
df1.set_index("Year",inplace=True)
df["Return"]=df["Return"]/100
yearly_max_drawdown = df.groupby("Year")["Return"].apply(lambda x: max_drawdown(x))
df1["Max_drawdown"]=yearly_max_drawdown
df1



In [ ]:
#spy etf rotation
#Going for losers
import pandas as pd
import numpy as np
from empyrical import max_drawdown
from calendar import monthrange

def monthly_returns(df):
    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index("Date",inplace=True)
    returns = {}
    for year in range(2019, 2023):
        for month in range(1, 13):
            start_date = pd.Timestamp(year=year, month=month, day=1) - pd.Timedelta(days=1)
            end_date = pd.Timestamp(year=year, month=month, day=monthrange(year, month)[1])
            start_price = df['Adj Close'].loc[:start_date].iloc[-1]
            end_price = df['Adj Close'].loc[:end_date].iloc[-1]
            returns[f"{year}-M{month:02d}"] = (end_price - start_price) / start_price
    return pd.Series(returns)

data = {etf: pd.read_csv(f"{etf}_weekly_return_detailed.csv") for etf in ['xlb', 'xle', 'xlf', 'xli', 'xlk', 'xlp', 'xlu', 'xlv', 'xly']}

monthly_returns_dict = {key: monthly_returns(df) for key, df in data.items()}
final_df = pd.DataFrame(monthly_returns_dict)
final_df.index.name = 'Date'

cash_w, cash_m, cash_l = 100, 100, 100
cash_monthly_w, cash_monthly_m, cash_monthly_l = [], [], []

for period in final_df.index[:-1]:
    sorted_returns = final_df.loc[period].sort_values()
    first_three = sorted_returns.iloc[0:3]
    middle_three = sorted_returns.iloc[3:6]
    last_three = sorted_returns.iloc[6:9]
    next_period = final_df.index[final_df.index.get_loc(period) + 1]

    cash_w = ((cash_w / 3) * (1 + final_df.loc[next_period][last_three.index])).sum()
    cash_m = ((cash_m / 3) * (1 + final_df.loc[next_period][middle_three.index])).sum()
    cash_l = ((cash_l / 3) * (1 + final_df.loc[next_period][first_three.index])).sum()

    cash_monthly_w.append((next_period, cash_w))
    cash_monthly_m.append((next_period, cash_m))
    cash_monthly_l.append((next_period, cash_l))

end_of_year_values = [value for date, value in cash_monthly_l if date.endswith('-M12')]
years = list(range(2001, 2024))
df1 = pd.DataFrame({'Year': years, 'Value': end_of_year_values})
initial_value = 100
df1['Annualized_Return'] = (df1['Value'] - df1['Value'].shift(1, fill_value=initial_value)) / df1['Value'].shift(1, fill_value=initial_value) * 100

daily_returns_df = pd.DataFrame()

for etf, df in data.items():
    df = df.reset_index()
    df['Date'] = pd.to_datetime(df['Date'])
    df.rename(columns={'Return': etf}, inplace=True)
    if daily_returns_df.empty:
        daily_returns_df = df[['Date', etf]]
    else:
        daily_returns_df = pd.merge(daily_returns_df, df[['Date', etf]], on='Date', how='outer')

daily_returns_df.set_index('Date', inplace=True)


result_df = pd.DataFrame()
for period in final_df.index:
    sorted_returns = final_df.loc[period].sort_values()
    middle = sorted_returns.iloc[0:3]

    temp_df = pd.DataFrame({
        'ETF1': [middle.index[0]],
        'ETF2': [middle.index[1]],
        'ETF3': [middle.index[2]]
    }, index=[period])

    result_df = pd.concat([result_df, temp_df])

final_result_df = pd.DataFrame()

for year in range(2001, 2024):
    for month in range(1, 13):
        period = f"{year}-M{month:02d}"
        if period not in result_df.index:
            continue

        etfs = result_df.loc[period].values

        next_month = month + 1
        next_year = year

        if next_month > 12:
            next_month = 1
            next_year = year + 1

        next_period_start = pd.Timestamp(year=next_year, month=next_month, day=1)
        next_period_end = pd.Timestamp(year=next_year, month=next_month, day=monthrange(next_year, next_month)[1])

        selected_returns = daily_returns_df[etfs].loc[next_period_start:next_period_end]
        avg_daily_returns = selected_returns.mean(axis=1)
        final_result_df = pd.concat([final_result_df, avg_daily_returns])

yearly_volatility = final_result_df.groupby(final_result_df.index.year).std() * np.sqrt(252)
df1.set_index("Year", inplace=True)
df1["Yearly_Volatility"]=yearly_volatility
df1["Sharpe_Ratio"]=df1["Annualized_Return"]/df1["Yearly_Volatility"]
final_result_df=final_result_df/100
yearly_max_drawdown = final_result_df.groupby(final_result_df.index.year).apply(lambda x: max_drawdown(x))
df1["Max_drawdown"]=yearly_max_drawdown
df2=df1
df2

In [ ]:
#spy etf rotation
#Going for middle three
import pandas as pd
import numpy as np
from empyrical import max_drawdown
from calendar import monthrange

def monthly_returns(df):
    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index("Date",inplace=True)
    returns = {}
    for year in range(2019, 2023):
        for month in range(1, 13):
            start_date = pd.Timestamp(year=year, month=month, day=1) - pd.Timedelta(days=1)
            end_date = pd.Timestamp(year=year, month=month, day=monthrange(year, month)[1])
            start_price = df['Adj Close'].loc[:start_date].iloc[-1]
            end_price = df['Adj Close'].loc[:end_date].iloc[-1]
            returns[f"{year}-M{month:02d}"] = (end_price - start_price) / start_price
    return pd.Series(returns)

data = {etf: pd.read_csv(f"{etf}_weekly_return_detailed.csv") for etf in ['xlb', 'xle', 'xlf', 'xli', 'xlk', 'xlp', 'xlu', 'xlv', 'xly']}

monthly_returns_dict = {key: monthly_returns(df) for key, df in data.items()}
final_df = pd.DataFrame(monthly_returns_dict)
final_df.index.name = 'Date'

cash_w, cash_m, cash_l = 100, 100, 100
cash_monthly_w, cash_monthly_m, cash_monthly_l = [], [], []

for period in final_df.index[:-1]:
    sorted_returns = final_df.loc[period].sort_values()
    first_three = sorted_returns.iloc[0:3]
    middle_three = sorted_returns.iloc[3:6]
    last_three = sorted_returns.iloc[6:9]
    next_period = final_df.index[final_df.index.get_loc(period) + 1]

    cash_w = ((cash_w / 3) * (1 + final_df.loc[next_period][last_three.index])).sum()
    cash_m = ((cash_m / 3) * (1 + final_df.loc[next_period][middle_three.index])).sum()
    cash_l = ((cash_l / 3) * (1 + final_df.loc[next_period][first_three.index])).sum()

    cash_monthly_w.append((next_period, cash_w))
    cash_monthly_m.append((next_period, cash_m))
    cash_monthly_l.append((next_period, cash_l))

end_of_year_values = [value for date, value in cash_monthly_m if date.endswith('-M12')]
years = list(range(2001, 2024))
df1 = pd.DataFrame({'Year': years, 'Value': end_of_year_values})
initial_value = 100
df1['Annualized_Return'] = (df1['Value'] - df1['Value'].shift(1, fill_value=initial_value)) / df1['Value'].shift(1, fill_value=initial_value) * 100

daily_returns_df = pd.DataFrame()

for etf, df in data.items():
    df = df.reset_index()
    df['Date'] = pd.to_datetime(df['Date'])
    df.rename(columns={'Return': etf}, inplace=True)
    if daily_returns_df.empty:
        daily_returns_df = df[['Date', etf]]
    else:
        daily_returns_df = pd.merge(daily_returns_df, df[['Date', etf]], on='Date', how='outer')

daily_returns_df.set_index('Date', inplace=True)


result_df = pd.DataFrame()
for period in final_df.index:
    sorted_returns = final_df.loc[period].sort_values()
    middle = sorted_returns.iloc[3:6]

    temp_df = pd.DataFrame({
        'ETF1': [middle.index[0]],
        'ETF2': [middle.index[1]],
        'ETF3': [middle.index[2]]
    }, index=[period])

    result_df = pd.concat([result_df, temp_df])

final_result_df = pd.DataFrame()

for year in range(2019, 2023):
    for month in range(1, 13):
        period = f"{year}-M{month:02d}"
        if period not in result_df.index:
            continue

        etfs = result_df.loc[period].values

        next_month = month + 1
        next_year = year

        if next_month > 12:
            next_month = 1
            next_year = year + 1

        next_period_start = pd.Timestamp(year=next_year, month=next_month, day=1)
        next_period_end = pd.Timestamp(year=next_year, month=next_month, day=monthrange(next_year, next_month)[1])

        selected_returns = daily_returns_df[etfs].loc[next_period_start:next_period_end]
        avg_daily_returns = selected_returns.mean(axis=1)
        final_result_df = pd.concat([final_result_df, avg_daily_returns])

yearly_volatility = final_result_df.groupby(final_result_df.index.year).std() * np.sqrt(252)
df1.set_index("Year", inplace=True)
df1["Yearly_Volatility"]=yearly_volatility
df1["Sharpe_Ratio"]=df1["Annualized_Return"]/df1["Yearly_Volatility"]
final_result_df=final_result_df/100
yearly_max_drawdown = final_result_df.groupby(final_result_df.index.year).apply(lambda x: max_drawdown(x))
df1["Max_drawdown"]=yearly_max_drawdown
df3=df1
df3

In [ ]:
#spy etf rotation
#Going for winners
import pandas as pd
import numpy as np
from empyrical import max_drawdown
from calendar import monthrange

def monthly_returns(df):
    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index("Date",inplace=True)
    returns = {}
    for year in range(2019, 2023):
        for month in range(1, 13):
            start_date = pd.Timestamp(year=year, month=month, day=1) - pd.Timedelta(days=1)
            end_date = pd.Timestamp(year=year, month=month, day=monthrange(year, month)[1])
            start_price = df['Adj Close'].loc[:start_date].iloc[-1]
            end_price = df['Adj Close'].loc[:end_date].iloc[-1]
            returns[f"{year}-M{month:02d}"] = (end_price - start_price) / start_price
    return pd.Series(returns)

data = {etf: pd.read_csv(f"{etf}_weekly_return_detailed.csv") for etf in ['xlb', 'xle', 'xlf', 'xli', 'xlk', 'xlp', 'xlu', 'xlv', 'xly']}

monthly_returns_dict = {key: monthly_returns(df) for key, df in data.items()}
final_df = pd.DataFrame(monthly_returns_dict)
final_df.index.name = 'Date'

cash_w, cash_m, cash_l = 100, 100, 100
cash_monthly_w, cash_monthly_m, cash_monthly_l = [], [], []

for period in final_df.index[:-1]:
    sorted_returns = final_df.loc[period].sort_values()
    first_three = sorted_returns.iloc[0:3]
    middle_three = sorted_returns.iloc[3:6]
    last_three = sorted_returns.iloc[6:9]
    next_period = final_df.index[final_df.index.get_loc(period) + 1]

    cash_w = ((cash_w / 3) * (1 + final_df.loc[next_period][last_three.index])).sum()
    cash_m = ((cash_m / 3) * (1 + final_df.loc[next_period][middle_three.index])).sum()
    cash_l = ((cash_l / 3) * (1 + final_df.loc[next_period][first_three.index])).sum()

    cash_monthly_w.append((next_period, cash_w))
    cash_monthly_m.append((next_period, cash_m))
    cash_monthly_l.append((next_period, cash_l))

end_of_year_values = [value for date, value in cash_monthly_w if date.endswith('-M12')]
years = list(range(2001, 2024))
df1 = pd.DataFrame({'Year': years, 'Value': end_of_year_values})
initial_value = 100
df1['Annualized_Return'] = (df1['Value'] - df1['Value'].shift(1, fill_value=initial_value)) / df1['Value'].shift(1, fill_value=initial_value) * 100

daily_returns_df = pd.DataFrame()

for etf, df in data.items():
    df = df.reset_index()
    df['Date'] = pd.to_datetime(df['Date'])
    df.rename(columns={'Return': etf}, inplace=True)
    if daily_returns_df.empty:
        daily_returns_df = df[['Date', etf]]
    else:
        daily_returns_df = pd.merge(daily_returns_df, df[['Date', etf]], on='Date', how='outer')

daily_returns_df.set_index('Date', inplace=True)


result_df = pd.DataFrame()
for period in final_df.index:
    sorted_returns = final_df.loc[period].sort_values()
    middle = sorted_returns.iloc[6:9]

    temp_df = pd.DataFrame({
        'ETF1': [middle.index[0]],
        'ETF2': [middle.index[1]],
        'ETF3': [middle.index[2]]
    }, index=[period])

    result_df = pd.concat([result_df, temp_df])

final_result_df = pd.DataFrame()

for year in range(2001, 2024):
    for month in range(1, 13):
        period = f"{year}-M{month:02d}"
        if period not in result_df.index:
            continue

        etfs = result_df.loc[period].values

        next_month = month + 1
        next_year = year

        if next_month > 12:
            next_month = 1
            next_year = year + 1

        next_period_start = pd.Timestamp(year=next_year, month=next_month, day=1)
        next_period_end = pd.Timestamp(year=next_year, month=next_month, day=monthrange(next_year, next_month)[1])

        selected_returns = daily_returns_df[etfs].loc[next_period_start:next_period_end]
        avg_daily_returns = selected_returns.mean(axis=1)
        final_result_df = pd.concat([final_result_df, avg_daily_returns])

yearly_volatility = final_result_df.groupby(final_result_df.index.year).std() * np.sqrt(252)
df1.set_index("Year", inplace=True)
df1["Yearly_Volatility"]=yearly_volatility
df1["Sharpe_Ratio"]=df1["Annualized_Return"]/df1["Yearly_Volatility"]
final_result_df=final_result_df/100
yearly_max_drawdown = final_result_df.groupby(final_result_df.index.year).apply(lambda x: max_drawdown(x))
df1["Max_drawdown"]=yearly_max_drawdown
df4=df1
df4